# RAG-Based Profile Matching — Experimentation & Analysis

This notebook walks through the complete RAG pipeline for matching job descriptions to resumes.

**Sections:**
1. Setup & Data Loading
2. Chunking Analysis
3. Embedding Exploration
4. Search Quality Evaluation
5. Full Pipeline Demo (JD → Matches)
6. Performance Metrics
7. Parameter Sensitivity Analysis

## 1. Setup & Data Loading

In [ ]:
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Import project modules
from config import *
from fs_tools import read_file, list_files
from resume_rag import ResumeChunker, EmbeddingGenerator, ResumeVectorStore, MetadataExtractor, ResumeRAGPipeline
from job_matcher import JobMatcher, JobDescriptionProcessor, MatchScorer

print('All modules loaded successfully!')

In [ ]:
# Load all resume files
resume_result = list_files(str(RESUME_DIR))
resume_files = [f for f in resume_result['files'] 
                if os.path.splitext(f['name'])[1].lower() in SUPPORTED_EXTENSIONS]

print(f'Total resume files: {len(resume_files)}')
print(f'\nFile format distribution:')

ext_counts = {}
for f in resume_files:
    ext = os.path.splitext(f['name'])[1].lower()
    ext_counts[ext] = ext_counts.get(ext, 0) + 1

for ext, count in sorted(ext_counts.items()):
    print(f'  {ext}: {count} files')

In [ ]:
# Load all resume text content
resumes = []
for f in resume_files:
    result = read_file(f['path'])
    if result['success']:
        resumes.append({
            'filename': f['name'],
            'path': f['path'],
            'text': result['content'],
            'size_bytes': f['size_bytes'],
            'char_count': len(result['content']),
        })

df_resumes = pd.DataFrame(resumes)
print(f'Successfully loaded: {len(df_resumes)} resumes')
print(f'\nResume text length statistics:')
print(df_resumes['char_count'].describe().to_string())

In [ ]:
# Visualize resume sizes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(len(df_resumes)), df_resumes['char_count'].values, color=sns.color_palette('husl', len(df_resumes)))
axes[0].set_yticks(range(len(df_resumes)))
axes[0].set_yticklabels([n.replace('resume_', '').replace('.txt', '').replace('.pdf', '').replace('.docx', '') 
                         for n in df_resumes['filename']], fontsize=8)
axes[0].set_xlabel('Character Count')
axes[0].set_title('Resume Text Length by Candidate')

axes[1].hist(df_resumes['char_count'], bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Resume Lengths')

plt.tight_layout()
plt.show()

## 2. Chunking Analysis

In [ ]:
# Analyze chunking behavior
chunker = ResumeChunker()

all_chunks = []
for _, row in df_resumes.iterrows():
    chunks = chunker.chunk_resume(row['text'], row['filename'])
    for chunk in chunks:
        chunk['text_length'] = len(chunk['text'])
    all_chunks.extend(chunks)

df_chunks = pd.DataFrame(all_chunks)
print(f'Total chunks generated: {len(df_chunks)}')
print(f'Average chunks per resume: {len(df_chunks) / len(df_resumes):.1f}')
print(f'\nChunk size statistics (characters):')
print(df_chunks['text_length'].describe().to_string())

In [ ]:
# Visualize chunk distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Section type distribution
section_counts = df_chunks['section'].value_counts()
colors = sns.color_palette('husl', len(section_counts))
axes[0].barh(section_counts.index, section_counts.values, color=colors)
axes[0].set_xlabel('Count')
axes[0].set_title('Chunks by Section Type')

# Chunk size distribution
axes[1].hist(df_chunks['text_length'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(x=CHUNK_MAX_CHARS, color='red', linestyle='--', label=f'Max ({CHUNK_MAX_CHARS})')
axes[1].set_xlabel('Chunk Size (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Chunk Sizes')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Show sample chunks from one resume
sample_resume = df_resumes.iloc[0]
sample_chunks = chunker.chunk_resume(sample_resume['text'], sample_resume['filename'])

print(f"Sample chunks from: {sample_resume['filename']}")
print(f"Total chunks: {len(sample_chunks)}\n")

for i, chunk in enumerate(sample_chunks):
    print(f"--- Chunk {i} [{chunk['section']}] ({len(chunk['text'])} chars) ---")
    print(chunk['text'][:200] + '...' if len(chunk['text']) > 200 else chunk['text'])
    print()

## 3. Embedding Exploration

In [ ]:
# Generate embeddings for all chunks
embedder = EmbeddingGenerator()

chunk_texts = df_chunks['text'].tolist()
print(f'Generating embeddings for {len(chunk_texts)} chunks...')

start = time.time()
embeddings = embedder.generate(chunk_texts)
embed_time = time.time() - start

print(f'Done! Time: {embed_time:.2f}s')
print(f'Embedding dimension: {len(embeddings[0])}')
print(f'Throughput: {len(chunk_texts)/embed_time:.1f} chunks/sec')

In [ ]:
# Dimensionality reduction with t-SNE for visualization
import numpy as np
from sklearn.manifold import TSNE

# Only use section chunks (not full_resume) for cleaner visualization
section_mask = df_chunks['chunk_type'] == 'section'
section_embeddings = np.array([embeddings[i] for i in range(len(embeddings)) if section_mask.iloc[i]])
section_df = df_chunks[section_mask].reset_index(drop=True)

print(f'Running t-SNE on {len(section_embeddings)} section embeddings...')
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(section_embeddings)-1))
coords = tsne.fit_transform(section_embeddings)

section_df['tsne_x'] = coords[:, 0]
section_df['tsne_y'] = coords[:, 1]

print('t-SNE complete!')

In [ ]:
# Plot t-SNE colored by section type
fig, ax = plt.subplots(figsize=(12, 8))

sections = section_df['section'].unique()
colors = sns.color_palette('husl', len(sections))

for i, section in enumerate(sections):
    mask = section_df['section'] == section
    ax.scatter(section_df.loc[mask, 'tsne_x'], section_df.loc[mask, 'tsne_y'],
               c=[colors[i]], label=section, alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.set_title('t-SNE Visualization of Resume Chunks (Colored by Section)', fontsize=14)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Compute pairwise similarity matrix between full resumes
from sklearn.metrics.pairwise import cosine_similarity

full_mask = df_chunks['chunk_type'] == 'full_resume'
full_embeddings = np.array([embeddings[i] for i in range(len(embeddings)) if full_mask.iloc[i]])
full_names = df_chunks[full_mask]['candidate_name'].tolist()

sim_matrix = cosine_similarity(full_embeddings)

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 12))
short_names = [n.split()[-1] if n else f'Resume {i}' for i, n in enumerate(full_names)]

sns.heatmap(sim_matrix, xticklabels=short_names, yticklabels=short_names,
            cmap='YlOrRd', annot=False, fmt='.2f', ax=ax,
            vmin=0.3, vmax=1.0)
ax.set_title('Cosine Similarity Between Full Resumes', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Search Quality Evaluation

In [ ]:
# Initialize the RAG pipeline
pipeline = ResumeRAGPipeline()

# Check if resumes are already indexed
stats = pipeline.store.get_stats()
if stats['total_chunks'] == 0:
    print('No resumes indexed yet. Indexing now...')
    pipeline.index_all_resumes()
else:
    print(f'Already indexed: {stats["total_resumes"]} resumes, {stats["total_chunks"]} chunks')

In [ ]:
# Test queries with expected top results
test_queries = [
    {
        'query': 'Python Django developer with AWS experience',
        'expected_top': ['Arjun Sharma', 'Harish Menon', 'Ankit Patel'],
    },
    {
        'query': 'Machine learning engineer with NLP and PyTorch',
        'expected_top': ['Rahul Verma', 'Amit Saxena', 'Fatima Sheikh'],
    },
    {
        'query': 'React and Node.js full-stack developer',
        'expected_top': ['Sneha Iyer', 'Varun Tiwari'],
    },
    {
        'query': 'Kubernetes DevOps engineer with Terraform',
        'expected_top': ['Deepak Gupta', 'Lakshmi Sundaram', 'Anjali Choudhary'],
    },
    {
        'query': 'Data engineer with Spark and Airflow',
        'expected_top': ['Vikram Reddy', 'Vivek Shankar'],
    },
]

print('Search Quality Test Results')
print('=' * 70)

quality_results = []

for test in test_queries:
    start = time.time()
    results = pipeline.search(test['query'], top_k=5)
    latency = (time.time() - start) * 1000
    
    top_names = [r['candidate_name'] for r in results]
    
    # Check how many expected results appear in top 5
    hits = sum(1 for exp in test['expected_top'] if exp in top_names)
    precision = hits / len(test['expected_top'])
    
    print(f"\nQuery: \"{test['query']}\"")
    print(f"  Expected: {test['expected_top']}")
    print(f"  Got (top 5): {top_names}")
    print(f"  Precision@5: {precision:.0%} ({hits}/{len(test['expected_top'])})")
    print(f"  Latency: {latency:.0f}ms")
    
    quality_results.append({
        'query': test['query'],
        'precision': precision,
        'latency_ms': latency,
        'top_1_match': top_names[0] if top_names else 'N/A',
    })

df_quality = pd.DataFrame(quality_results)
print(f"\n{'='*70}")
print(f"Average Precision@5: {df_quality['precision'].mean():.0%}")
print(f"Average Latency: {df_quality['latency_ms'].mean():.0f}ms")

In [ ]:
# Visualize search quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision bar chart
short_queries = [q[:30] + '...' for q in df_quality['query']]
colors = ['green' if p >= 0.67 else 'orange' if p >= 0.33 else 'red' for p in df_quality['precision']]
axes[0].barh(short_queries, df_quality['precision'], color=colors, edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Precision@5')
axes[0].set_title('Retrieval Precision by Query')
axes[0].set_xlim(0, 1.1)

# Latency bar chart
axes[1].barh(short_queries, df_quality['latency_ms'], color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Latency (ms)')
axes[1].set_title('Search Latency by Query')

plt.tight_layout()
plt.show()

## 5. Full Pipeline Demo — JD → Matches

In [ ]:
# Load job descriptions
jd_dir = str(JD_DIR)
jd_files = sorted([f for f in os.listdir(jd_dir) if f.endswith('.txt')])

print(f'Job Descriptions ({len(jd_files)}):')
for f in jd_files:
    with open(os.path.join(jd_dir, f), 'r') as fh:
        first_line = fh.readline().strip()
    print(f'  • {f}: {first_line}')

In [ ]:
# Run full matching pipeline for all JDs
matcher = JobMatcher()

all_match_results = []

for jd_file in jd_files:
    jd_path = os.path.join(jd_dir, jd_file)
    result = matcher.match_file(jd_path, top_k=10)
    all_match_results.append(result)
    print()

In [ ]:
# Display results as a summary table
summary_rows = []
for result in all_match_results:
    title = result['parsed_jd']['title']
    for match in result['top_matches'][:5]:  # Top 5 per JD
        summary_rows.append({
            'Job Title': title,
            'Rank': match['rank'],
            'Candidate': match['candidate_name'],
            'Score': match['match_score'],
            'Semantic': match['score_breakdown']['semantic_similarity'],
            'Skill': match['score_breakdown']['skill_match'],
            'Experience': match['score_breakdown']['experience_match'],
            'Matched Skills': ', '.join(match['matched_skills'][:4]),
        })

df_summary = pd.DataFrame(summary_rows)
print('Top 5 Matches per Job Description')
print('=' * 100)
print(df_summary.to_string(index=False))

In [ ]:
# Visualize match scores across JDs
fig, ax = plt.subplots(figsize=(14, 6))

job_titles = df_summary['Job Title'].unique()
colors = sns.color_palette('husl', len(job_titles))

for i, jt in enumerate(job_titles):
    jt_data = df_summary[df_summary['Job Title'] == jt]
    short_title = jt[:25] + '...' if len(jt) > 25 else jt
    ax.scatter(jt_data['Rank'], jt_data['Score'], 
               c=[colors[i]], label=short_title, s=100, edgecolors='black', alpha=0.8)

ax.set_xlabel('Rank')
ax.set_ylabel('Match Score (0-100)')
ax.set_title('Match Score Distribution Across Job Descriptions')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

In [ ]:
# Show detailed results for one JD
demo_result = all_match_results[0]  # Senior Python Backend
print(f"Detailed Results: {demo_result['parsed_jd']['title']}")
print('=' * 80)

for match in demo_result['top_matches'][:5]:
    print(f"\n  #{match['rank']} {match['candidate_name']} — Score: {match['match_score']}/100")
    bd = match['score_breakdown']
    print(f"     Semantic: {bd['semantic_similarity']:.0f} | Skill: {bd['skill_match']:.0f} | "
          f"Exp: {bd['experience_match']:.0f} | Edu: {bd['education_match']:.0f}")
    print(f"     Matched: {', '.join(match['matched_skills'][:6])}")
    if match['missing_skills']:
        print(f"     Missing: {', '.join(match['missing_skills'][:3])}")
    print(f"     Reasoning: {match['reasoning'][:150]}...")

In [ ]:
# Score breakdown comparison for top matches
demo_matches = demo_result['top_matches'][:5]
names = [m['candidate_name'].split()[-1] for m in demo_matches]
semantic = [m['score_breakdown']['semantic_similarity'] for m in demo_matches]
skill = [m['score_breakdown']['skill_match'] for m in demo_matches]
exp = [m['score_breakdown']['experience_match'] for m in demo_matches]
edu = [m['score_breakdown']['education_match'] for m in demo_matches]

x = range(len(names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar([i - 1.5*width for i in x], semantic, width, label='Semantic', color='#4C72B0')
bars2 = ax.bar([i - 0.5*width for i in x], skill, width, label='Skill Match', color='#55A868')
bars3 = ax.bar([i + 0.5*width for i in x], exp, width, label='Experience', color='#C44E52')
bars4 = ax.bar([i + 1.5*width for i in x], edu, width, label='Education', color='#8172B2')

ax.set_xlabel('Candidate')
ax.set_ylabel('Score Component (0-100)')
ax.set_title(f"Score Breakdown — {demo_result['parsed_jd']['title']}")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

## 6. Performance Metrics

In [ ]:
# Measure indexing performance
print('Performance Metrics')
print('=' * 50)

# Embedding generation latency
test_texts = [r['text'] for r in resumes[:5]]
embed_times = []
for text in test_texts:
    start = time.time()
    embedder.generate([text])
    embed_times.append((time.time() - start) * 1000)

print(f'\nEmbedding Generation:')
print(f'  Avg latency per text: {sum(embed_times)/len(embed_times):.1f}ms')
print(f'  Min: {min(embed_times):.1f}ms | Max: {max(embed_times):.1f}ms')

# Search latency
search_times = []
test_search_queries = [
    'Python developer',
    'Machine learning engineer NLP',
    'Full-stack React Node.js TypeScript',
    'DevOps Kubernetes Terraform AWS',
    'Data engineer Spark Airflow SQL',
]

for query in test_search_queries:
    start = time.time()
    pipeline.search(query, top_k=10)
    search_times.append((time.time() - start) * 1000)

print(f'\nSemantic Search (top-10):')
print(f'  Avg latency: {sum(search_times)/len(search_times):.1f}ms')
print(f'  Min: {min(search_times):.1f}ms | Max: {max(search_times):.1f}ms')

# Full match latency (from previous runs)
match_latencies = [r['performance_metrics']['retrieval_latency_ms'] for r in all_match_results]
print(f'\nFull Match Pipeline (JD → Results):')
print(f'  Avg latency: {sum(match_latencies)/len(match_latencies):.0f}ms')
print(f'  Min: {min(match_latencies)}ms | Max: {max(match_latencies)}ms')

In [ ]:
# Latency visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Search latency
axes[0].bar(range(len(search_times)), search_times, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xticks(range(len(search_times)))
axes[0].set_xticklabels([q[:20]+'...' for q in test_search_queries], rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Semantic Search Latency')
axes[0].axhline(y=500, color='red', linestyle='--', alpha=0.5, label='500ms threshold')
axes[0].legend()

# Full pipeline latency
jd_labels = [r['parsed_jd']['title'][:20]+'...' for r in all_match_results]
axes[1].bar(range(len(match_latencies)), match_latencies, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(len(match_latencies)))
axes[1].set_xticklabels(jd_labels, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Full Match Pipeline Latency')

plt.tight_layout()
plt.show()

## 7. Parameter Sensitivity Analysis

In [ ]:
# Test effect of top-K on result quality
test_query = 'Senior Python backend developer with Django and AWS experience'
k_values = [3, 5, 10, 15, 20]

k_results = []
for k in k_values:
    start = time.time()
    results = pipeline.search(test_query, top_k=k)
    latency = (time.time() - start) * 1000
    
    similarities = [r['similarity'] for r in results]
    k_results.append({
        'K': k,
        'actual_results': len(results),
        'avg_similarity': sum(similarities) / len(similarities) if similarities else 0,
        'min_similarity': min(similarities) if similarities else 0,
        'max_similarity': max(similarities) if similarities else 0,
        'latency_ms': latency,
    })

df_k = pd.DataFrame(k_results)
print('Effect of Top-K on Results')
print(df_k.to_string(index=False))

In [ ]:
# Visualize K sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_k['K'], df_k['avg_similarity'], 'o-', color='steelblue', label='Avg Similarity', linewidth=2)
axes[0].fill_between(df_k['K'], df_k['min_similarity'], df_k['max_similarity'], alpha=0.2, color='steelblue')
axes[0].plot(df_k['K'], df_k['min_similarity'], '--', color='gray', alpha=0.5, label='Min Similarity')
axes[0].set_xlabel('Top-K')
axes[0].set_ylabel('Cosine Similarity')
axes[0].set_title('Retrieval Quality vs Top-K')
axes[0].legend()

axes[1].plot(df_k['K'], df_k['latency_ms'], 'o-', color='coral', linewidth=2)
axes[1].set_xlabel('Top-K')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Search Latency vs Top-K')

plt.tight_layout()
plt.show()

In [ ]:
# Test score weight sensitivity
weight_configs = [
    {'name': 'Default (40/30/15/15)', 'semantic': 0.40, 'skill': 0.30, 'exp': 0.15, 'edu': 0.15},
    {'name': 'Semantic Heavy (60/20/10/10)', 'semantic': 0.60, 'skill': 0.20, 'exp': 0.10, 'edu': 0.10},
    {'name': 'Skill Heavy (20/50/15/15)', 'semantic': 0.20, 'skill': 0.50, 'exp': 0.15, 'edu': 0.15},
    {'name': 'Experience Heavy (30/25/30/15)', 'semantic': 0.30, 'skill': 0.25, 'exp': 0.30, 'edu': 0.15},
    {'name': 'Equal (25/25/25/25)', 'semantic': 0.25, 'skill': 0.25, 'exp': 0.25, 'edu': 0.25},
]

# Use the first JD result for comparison
demo_jd = all_match_results[0]
print(f"Weight Sensitivity for: {demo_jd['parsed_jd']['title']}")
print('=' * 80)

for config in weight_configs:
    print(f"\n{config['name']}:")
    for match in demo_jd['top_matches'][:5]:
        bd = match['score_breakdown']
        score = (
            config['semantic'] * bd['semantic_similarity'] +
            config['skill'] * bd['skill_match'] +
            config['exp'] * bd['experience_match'] +
            config['edu'] * bd['education_match']
        )
        print(f"  {match['candidate_name']:20s} — Score: {score:.0f}")

In [ ]:
# Final summary
store_stats = pipeline.store.get_stats()

print('\n' + '=' * 60)
print('  RAG Profile Matching — Final Summary')
print('=' * 60)
print(f'\n  Dataset:')
print(f'    Resumes indexed:      {store_stats["total_resumes"]}')
print(f'    Total chunks:         {store_stats["total_chunks"]}')
print(f'    Job descriptions:     {len(jd_files)}')
print(f'\n  Quality:')
print(f'    Avg Precision@5:      {df_quality["precision"].mean():.0%}')
print(f'    Avg Search Latency:   {sum(search_times)/len(search_times):.0f}ms')
print(f'    Avg Match Latency:    {sum(match_latencies)/len(match_latencies):.0f}ms')
print(f'\n  Configuration:')
print(f'    Embedding Model:      {HF_EMBEDDING_MODEL}')
print(f'    Embedding Dimension:  {embedder.dim}')
print(f'    Chunk Max Size:       {CHUNK_MAX_CHARS} chars')
print(f'    Score Weights:        Semantic={WEIGHT_SEMANTIC}, Skill={WEIGHT_SKILL}, Exp={WEIGHT_EXPERIENCE}, Edu={WEIGHT_EDUCATION}')
print()